# Detección automática de eventos adversos hospitalarios en epicrisis

**Curso MIA-10 — Procesamiento del Lenguaje Natural**
Maestría en Inteligencia Artificial · Universidad Nacional de Ingeniería
Docente: Dr. Wester Zela Moraya · Autor: Mg. Carlos Pérez Pérez

---

## Qué resuelve este trabajo

La notificación de eventos adversos en EsSalud es manual y voluntaria, con un
subregistro cercano al **72 %** (unos 37,000 eventos no reportados al año). La
información existe —está escrita en las epicrisis— pero en texto libre, no
explotable automáticamente.

**Pregunta de investigación:** ¿es posible detectar automáticamente, desde el
texto libre de los resúmenes de alta, los eventos adversos ocurridos durante la
hospitalización, y clasificarlos según la taxonomía normativa, con una
fiabilidad comparable a la del juicio experto?

## Cómo está organizado

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga de artefactos |
| 2 | Corpus: composición y prevalencia |
| 3 | Validez del etiquetado: los siete modos de fallo |
| 4 | El confusor de época (aprendizaje por atajo) |
| 5 | Etapa 1 — detección binaria |
| 6 | Etapa 2 — naturaleza y evaluación en cascada |
| 7 | Ranking de desempeño de los modelos |
| 8 | Validación con evaluador independiente (kappa) |
| 9 | Transferencia al español con etiqueta de oro |
| 10 | Conclusiones |

> **Nota sobre reproducibilidad.** El entrenamiento completo del pipeline exige
> ~13 GB de texto de MIMIC-IV (acceso credencializado por PhysioNet) y varias
> horas de cómputo. Este notebook **carga los artefactos ya generados** por los
> scripts del pipeline y reproduce el análisis y las métricas en minutos. Cada
> sección indica el script que produjo su artefacto.

## 1. Configuración y carga de artefactos

Los resultados provienen de cinco scripts del pipeline:

| Script | Produce |
|---|---|
| `fase9_modelo_final.py` | modelo final + métricas de detección y naturaleza |
| `fase10_metricas_corregidas.py` | IC agrupado por paciente y evaluación en cascada |
| `fase11_finetuning_transformers.py` | ajuste fino de Bio_ClinicalBERT y BioBERT |
| `fase6_concordancia_kappa.py` | concordancia inter-observador |
| `oe5_ersp_preparar.py` | corpus español con etiqueta de oro |

> **Portabilidad.** El notebook no lleva rutas fijas: localiza la carpeta de
> resultados buscando hacia arriba desde su propia ubicación, y admite
> sobrescribirla con la variable de entorno `PLN_RESULTADOS`. Si un artefacto no
> está disponible, la celda correspondiente lo informa y continúa en lugar de
> abortar, de modo que el notebook se ejecuta de principio a fin aunque solo
> haya una parte de los resultados.

In [1]:
import json, os
from pathlib import Path
import numpy as np, pandas as pd

def localizar_resultados():
    # 1) variable de entorno, 2) resultados junto al notebook,
    # 3) busqueda hacia arriba, 4) ruta de desarrollo como ultimo recurso
    if os.environ.get("PLN_RESULTADOS"):
        return Path(os.environ["PLN_RESULTADOS"])
    aqui = Path.cwd()
    for base in [aqui, *aqui.parents]:
        for cand in (base / "resultados",
                     base / "04_pipeline_codigo" / "datos_intermedios"):
            if cand.is_dir():
                return cand
    return Path(r"T:/MIMIC/tesis/04_pipeline_codigo/datos_intermedios")

D = localizar_resultados()
print(f"carpeta de resultados: {D}")
print(f"  existe: {D.is_dir()}")

def cargar(ruta):
    f = D / ruta
    if not f.exists():
        return None
    try:
        return json.loads(f.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"  [!] {ruta}: {type(e).__name__}")
        return None

f9  = cargar("fase9_final/resultados_finales.json")      # deteccion + naturaleza
f10 = cargar("fase9_final/metricas_corregidas.json")     # IC agrupado + cascada
f11 = cargar("fase11/resultados_transformers.json")      # transformers
kap = cargar("fase6_concordancia/concordancia.json")     # kappa
oe5 = cargar("oe5_ersp/informe_oe5.json")                # espanol

print("\nartefactos:")
for n, a in [("fase9", f9), ("fase10", f10), ("fase11", f11),
             ("kappa", kap), ("oe5", oe5)]:
    print(f"  {n:<8} {'OK' if a else 'NO DISPONIBLE'}")

def falta(*art):
    """Aviso uniforme cuando un artefacto no esta: la celda informa y sigue."""
    print("Artefacto no disponible en esta maquina:", ", ".join(art))
    print("Los resultados de esta seccion figuran en el informe (paper/main.pdf).")
    return True

carpeta de resultados: T:\MIMIC\tesis\04_pipeline_codigo\datos_intermedios
  existe: True

artefactos:
  fase9    OK
  fase10   OK
  fase11   OK
  kappa    OK
  oe5      OK


## 2. Corpus: composición y prevalencia

Fuente: **MIMIC-IV-Note v2.2** (PhysioNet), 331,793 resúmenes de alta del Beth
Israel Deaconess Medical Center (2008–2019).

El corpus de modelado se acotó por memoria: vectorizar n-gramas de carácter
sobre más de 100,000 notas desborda la RAM disponible. El techo práctico
verificado fue de 70,000 notas.

Un punto importante: la **prevalencia poblacional** (proporción de
hospitalizaciones con evento codificado) es muy distinta de la del conjunto de
prueba, que está balanceado. Esa diferencia es la que obliga a reajustar el VPP.

In [2]:
if not f9:
    falta("fase9_final/resultados_finales.json")
else:
    c = f9["corpus"]
    print(f"epicrisis en el corpus de modelado : {c['epicrisis']:,}")
    print(f"  de ellas positivas              : {c['positivas']:,} "
          f"({c['positivas']/c['epicrisis']:.1%} del corpus)")
    print(f"prevalencia POBLACIONAL real      : {c['prevalencia_real']:.2%}")
    print(f"  {c['universo_positivo']:,} hospitalizaciones CON evento")
    print(f"  sobre un universo de {c['universo_positivo']/c['prevalencia_real']:,.0f}")
    print()
    e = c["emparejamiento_epoca"]
    print("Emparejamiento por epoca (control del confusor, ver seccion 4):")
    print(f"  positivos era CIE-10: {e['positivos_era10']:.2%}")
    print(f"  negativos era CIE-10: {e['negativos_era10']:.2%}")
    print(f"  diferencia          : {abs(e['positivos_era10']-e['negativos_era10']):.4%}")

epicrisis en el corpus de modelado : 70,000
  de ellas positivas              : 37,692 (53.8% del corpus)
prevalencia POBLACIONAL real      : 20.12%
  109,775 hospitalizaciones CON evento
  sobre un universo de 545,601

Emparejamiento por epoca (control del confusor, ver seccion 4):
  positivos era CIE-10: 28.92%
  negativos era CIE-10: 28.92%
  diferencia          : 0.0000%


## 3. Validez del etiquetado: los siete modos de fallo

El etiquetado se construyó por **supervisión débil** (Ratner et al., Snorkel):
reglas sobre códigos diagnósticos y patrones textuales generan las etiquetas
sin anotación manual exhaustiva. El precio de esa escala es que la calidad del
corpus queda supeditada a la calidad de las reglas, de modo que hay que
verificarla explícitamente.

Un protocolo sistemático de control identificó siete modos de fallo. La celda
siguiente los lista con su efecto medido.

In [3]:
modos = pd.DataFrame([
 ("1. Muestreo no declarado (solo 9% del corpus)", "301,793 epicrisis sin revisar"),
 ("2. re.DOTALL: el comodin cruzaba la nota entera", "-63.7% de detecciones"),
 ("3. CIE-10 con punto vs MIMIC sin punto",         "corpus positivo x267"),
 ("4. Ausencia de clase negativa",                  "abstencion 6/6 tras corregir"),
 ("5. Confusor de epoca CIE-9/CIE-10",              "ver seccion 4 (el mas grave)"),
 ("6. 43/223 codigos eran reglas muertas (OMS!=CM)","Medicacion x68 (209->14,295)"),
 ("7. Percentiles degenerados con n pequeno",       "inversion de prioridad"),
], columns=["Modo de fallo", "Efecto medido"])
modos.style.hide(axis="index")

Modo de fallo,Efecto medido
1. Muestreo no declarado (solo 9% del corpus),"301,793 epicrisis sin revisar"
2. re.DOTALL: el comodin cruzaba la nota entera,-63.7% de detecciones
3. CIE-10 con punto vs MIMIC sin punto,corpus positivo x267
4. Ausencia de clase negativa,abstencion 6/6 tras corregir
5. Confusor de epoca CIE-9/CIE-10,ver seccion 4 (el mas grave)
6. 43/223 codigos eran reglas muertas (OMS!=CM),"Medicacion x68 (209->14,295)"
7. Percentiles degenerados con n pequeno,inversion de prioridad


### Evidencia decisiva del modo 2

El comodín `.*` con `re.DOTALL` hacía que un patrón pudiera abarcar la epicrisis
completa, generando coincidencias espurias entre secciones sin relación.

La prueba de que el problema era el **alcance** y no la especificidad de los
patrones: al acotarlo, los patrones **con** `.*` cayeron un 84 %, mientras que
los patrones **sin** comodín quedaron exactamente iguales.

In [4]:
prueba = pd.DataFrame({
    "Tipo de patron": ["Con comodin .*", "Sin comodin"],
    "Antes": [42330, 13189],
    "Despues": [6735, 13189],
})
prueba["Variacion"] = (prueba.Despues/prueba.Antes - 1).map("{:.1%}".format)
print(prueba.to_string(index=False))
print("\nLos patrones sin comodin no cambian: el defecto era el ALCANCE.")

Tipo de patron  Antes  Despues Variacion
Con comodin .*  42330     6735    -84.1%
   Sin comodin  13189    13189      0.0%

Los patrones sin comodin no cambian: el defecto era el ALCANCE.


## 4. El confusor de época: un caso de aprendizaje por atajo

Este es el hallazgo metodológico central.

El mapeo inicial contenía **solo códigos CIE-10**. Como MIMIC-IV abarca
2008–2019 y la transición CIE-9→CIE-10 ocurrió en 2015, toda hospitalización
anterior resultaba negativa **por construcción**:

- negativos: 78.97 % de la era CIE-9
- positivos: 100 % de la era CIE-10

El clasificador no aprendió a reconocer eventos adversos: aprendió a distinguir
**la plantilla documental de cada época**. El rasgo de mayor peso era
`palabra__rdwsd`, un artefacto de cabecera de laboratorio sin ningún contenido
clínico.

Es un caso de libro de *shortcut learning* (Geirhos et al., 2020), análogo al
detector de neumonía de Zech et al. (2018) que había aprendido a reconocer el
hospital de procedencia por marcas en la radiografía.

In [5]:
if not f9:
    falta("fase9_final/resultados_finales.json")
else:
    d = f9["etapa1_deteccion"]
    versiones = pd.DataFrame([
        ("Inicial (INVALIDA, no se cita)", 0.917, 0.973, 0.433),
        ("Reevaluada con emparejamiento",  0.694, 0.904, 0.171),
        ("Final (7 modos corregidos)",
         d["especificidad"], d["auc"], d["vpp_prevalencia_real"]),
    ], columns=["Version", "Especificidad", "AUC", "VPP"])
    print(versiones.to_string(index=False))
    print()
    print("Leccion: un AUC de 0.973 puede sostenerse en una senal espuria.")
    top = f9.get("rasgos_top") or f9.get("rasgos") or []
    if top:
        print("\nRasgos de mayor peso tras corregir (del propio modelo):")
        for r in top[:8]:
            nom, w = (r[0], r[1]) if isinstance(r, (list, tuple)) else (r, None)
            print(f"  {nom}" + (f"   (peso {w})" if w is not None else ""))
    else:
        print("(el detalle de rasgos no figura en este artefacto)")

                       Version  Especificidad    AUC   VPP
Inicial (INVALIDA, no se cita)         0.9170 0.9730 0.433
 Reevaluada con emparejamiento         0.6940 0.9040 0.171
    Final (7 modos corregidos)         0.7699 0.8426 0.455

Leccion: un AUC de 0.973 puede sostenerse en una senal espuria.
(el detalle de rasgos no figura en este artefacto)


## 5. Etapa 1 — detección binaria

Vectorización TF-IDF (palabra + carácter) con LinearSVC balanceado.

Tres decisiones de protocolo condicionan la validez:

1. **Partición por paciente** (`GroupShuffleSplit` sobre `subject_id`), con
   aserción explícita de que no hay solape. Particionar por nota inflaría las
   métricas, porque un mismo paciente aporta notas con vocabulario compartido.
2. **Vectorizador ajustado solo en entrenamiento.**
3. **Intervalos por bootstrap agrupado por paciente**, no por nota: remuestrear
   notas viola la independencia y estrecha artificialmente los intervalos.

In [6]:
if not (f9 and f10):
    falta("resultados_finales.json", "metricas_corregidas.json")
else:
    e1 = f9["etapa1_deteccion"]
    ic = f10["A_intervalos"]
    tabla = pd.DataFrame([
        ("Sensibilidad",  e1["sensibilidad"],  ic["sensibilidad"]["ic95_por_paciente"]),
        ("Especificidad", e1["especificidad"], ic["especificidad"]["ic95_por_paciente"]),
        ("AUC",           e1["auc"],           ic["AUC"]["ic95_por_paciente"]),
    ], columns=["Metrica", "Valor", "IC95 (agrupado por paciente)"])
    print(tabla.to_string(index=False))
    print()
    m = e1["matriz"]
    print(f"Matriz de confusion: VP={m['vp']:,}  FP={m['fp']:,}  "
          f"FN={m['fn']:,}  VN={m['vn']:,}")
    print(f"VPP crudo en el test (balanceado): {m['vp']/(m['vp']+m['fp']):.3f}")
    print(f"VPP reajustado a prevalencia real: "
          f"{e1['vpp_prevalencia_real']:.3f}  <-- el operativo")

      Metrica  Valor IC95 (agrupado por paciente)
 Sensibilidad 0.7623             [0.7514, 0.7729]
Especificidad 0.7699             [0.7591, 0.7809]
          AUC 0.8426             [0.8361, 0.8493]

Matriz de confusion: VP=5,761  FP=1,502  FN=1,796  VN=5,026
VPP crudo en el test (balanceado): 0.793
VPP reajustado a prevalencia real: 0.455  <-- el operativo


### Por qué se reajusta el VPP

El conjunto de prueba está balanceado, así que su VPP bruto es optimista. La
cifra que importa en operación es el VPP a la **prevalencia poblacional**,
obtenido por el teorema de Bayes: determina cuántas revisiones improductivas
generaría el sistema en un servicio de calidad real.

In [7]:
def vpp(sens, esp, prev):
    """Teorema de Bayes: VPP = (S*p) / (S*p + (1-E)*(1-p))."""
    return (sens*prev) / (sens*prev + (1-esp)*(1-prev))

if not f9:
    falta("resultados_finales.json")
else:
    e1 = f9["etapa1_deteccion"]
    s, e_, p = e1["sensibilidad"], e1["especificidad"], f9["corpus"]["prevalencia_real"]
    print(f"VPP({s:.3f}, {e_:.3f}, prev={p:.3f}) = {vpp(s, e_, p):.4f}")
    print(f"reportado en el pipeline           = {e1['vpp_prevalencia_real']:.4f}")
    print("\nSensibilidad del VPP a la prevalencia:")
    for pr in [0.05, 0.10, 0.2012, 0.35, 0.50]:
        marca = "  <-- la real" if abs(pr - p) < 1e-4 else ""
        print(f"  prevalencia {pr:>6.2%}  ->  VPP {vpp(s, e_, pr):.3f}{marca}")

VPP(0.762, 0.770, prev=0.201) = 0.4549
reportado en el pipeline           = 0.4550

Sensibilidad del VPP a la prevalencia:
  prevalencia  5.00%  ->  VPP 0.148
  prevalencia 10.00%  ->  VPP 0.269
  prevalencia 20.12%  ->  VPP 0.455  <-- la real
  prevalencia 35.00%  ->  VPP 0.641
  prevalencia 50.00%  ->  VPP 0.768


### Abstención ante texto trivial

Un detector sin clase negativa marca como positivo casi cualquier texto. Tras
incorporarla, se comprobó el comportamiento ante entradas sin contenido clínico.

In [8]:
ab = pd.DataFrame(e1["abstencion"])
ab["detecta"] = ab.detecta.map({True: "SI (mal)", False: "se abstiene (bien)"})
print(ab.to_string(index=False))
print(f"\nAbstiene en {(pd.DataFrame(e1['abstencion']).detecta == False).sum()}"
      f"/{len(e1['abstencion'])} casos triviales.")

                                               texto  margen            detecta
                                                   .  -2.284 se abstiene (bien)
                                                      -1.249 se abstiene (bien)
                                                  ok  -1.180 se abstiene (bien)
                      Paciente estable, sin novedad.  -0.978 se abstiene (bien)
Routine follow up visit. Vital signs stable. No acut  -0.876 se abstiene (bien)
Patient admitted for elective knee replacement. Unev  -0.588 se abstiene (bien)

Abstiene en 6/6 casos triviales.


## 6. Etapa 2 — naturaleza y evaluación en cascada

La Etapa 2 clasifica la naturaleza del evento sobre las notas positivas.

Evaluarla sobre positivos **de referencia** supone un detector perfecto y
sobreestima el rendimiento real. La cifra honesta es la **cascada** completa:
texto → detección → naturaleza, donde los errores se acumulan.

Una trampa que hubo que evitar: las dos etapas se particionaron por separado,
así que un paciente del test de la Etapa 1 podía estar en el entrenamiento de la
Etapa 2. La cascada se evalúa solo sobre pacientes no vistos por **ninguna**.

In [9]:
cas = f10["B_cascada"]
comp = pd.DataFrame([
 ("Etapa 2 aislada (positivos de referencia)",
  cas["etapa2_aislada"]["f1_micro"], cas["etapa2_aislada"]["f1_macro"]),
 ("CASCADA real (Etapa 1 -> Etapa 2)",
  cas["cascada"]["f1_micro"], cas["cascada"]["f1_macro"]),
], columns=["Evaluacion", "F1-micro", "F1-macro"])
print(comp.to_string(index=False))
d = cas["cascada"]["f1_micro"]/cas["etapa2_aislada"]["f1_micro"] - 1
print(f"\nCaida al encadenar: {d:.1%} en F1-micro")
print(f"Evaluado sobre {cas['n_notas_limpias']:,} notas limpias "
      f"({cas['pct_test']:.1%} del test; el resto se descarto por fuga cruzada)")

                               Evaluacion  F1-micro  F1-macro
Etapa 2 aislada (positivos de referencia)    0.7516    0.5126
        CASCADA real (Etapa 1 -> Etapa 2)    0.4926    0.3626

Caida al encadenar: -34.5% en F1-micro
Evaluado sobre 6,655 notas limpias (47.2% del test; el resto se descarto por fuga cruzada)


## 7. Ranking de desempeño de los modelos

Todos los modelos se evalúan sobre la **misma partición estratificada**
(semilla 42), de modo que las cifras son comparables entre sí.

Se incluyen dos familias:

- **Léxica (línea base):** TF-IDF + LinearSVC.
- **Transformers clínicos:** Bio_ClinicalBERT y BioBERT, con ajuste fino
  completo sobre GPU.

Y un tercer experimento de control: TF-IDF entrenado sobre el texto **truncado**
a la misma ventana que ve el transformer. Sirve para separar dos explicaciones
que suelen confundirse — *"el transformer es peor"* frente a *"el transformer
ve mucho menos texto"*.

In [10]:
if f11 and f11.get("resultados"):
    R = f11["resultados"]
    filas = [{"Modelo": k, **{m: v[m] for m in
              ("exactitud", "f1_macro", "f1_micro", "kappa") if m in v}}
             for k, v in R.items() if "error" not in v]
    rk = pd.DataFrame(filas).sort_values("f1_macro", ascending=False)
    rk.insert(0, "#", range(1, len(rk)+1))
    print("RANKING DE DESEMPENO (ordenado por F1-macro)")
    print(rk.to_string(index=False))

    v = f11["ventana"]
    print(f"\nVentana de {f11['config']['max_len']} tokens: cubre el "
          f"{v['cobertura_media']:.1%} del documento")
    print(f"(mediana de {v['tokens_mediana']:,} tokens por nota)")
else:
    print("fase11 aun no disponible: ejecutar fase11_finetuning_transformers.py")

RANKING DE DESEMPENO (ordenado por F1-macro)
 #                                                 Modelo  exactitud  f1_macro  f1_micro  kappa
 1                    TF-IDF + LinearSVC (texto completo)     0.7149    0.4592    0.7149 0.5443
 2       TF-IDF + LinearSVC sin balanceo (texto completo)     0.7109    0.3909    0.7109 0.5132
 3              Bio_ClinicalBERT (fine-tuning, ponderado)     0.5429    0.3541    0.5429 0.3252
 4              TF-IDF + LinearSVC (truncado a 1150 car.)     0.5746    0.3301    0.5746 0.3148
 5 TF-IDF + LinearSVC sin balanceo (truncado a 1150 car.)     0.5890    0.2747    0.5890 0.2951
 6                         Bio_ClinicalBERT (fine-tuning)     0.6072    0.2487    0.6072 0.3438
 7                                  BioBERT (fine-tuning)     0.5971    0.2097    0.5971 0.3182

Ventana de 256 tokens: cubre el 9.0% del documento
(mediana de 3,148 tokens por nota)


In [11]:
if f11 and f11.get("resultados"):
    R = f11["resultados"]
    a = R.get("TF-IDF + LinearSVC (texto completo)")
    b = [v for k, v in R.items() if "truncado" in k]
    if a and b:
        b = b[0]
        print("EFECTO AISLADO DE LA VENTANA DE CONTEXTO")
        print("(misma arquitectura, mismos datos, solo cambia cuanto texto ve)\n")
        for m in ("exactitud", "f1_macro", "kappa"):
            d = b[m] - a[m]
            print(f"  {m:<12} completo {a[m]:.3f} -> truncado {b[m]:.3f} "
                  f"({d:+.3f}, {d/a[m]:+.1%})")
        print("\nConclusion: parte sustancial de la desventaja del transformer")
        print("no viene de la arquitectura sino de la ventana de 512 tokens.")
        print("Coincide con Li et al. (2022): Clinical-Longformer, que extiende")
        print("la ventana a 4,096, supera consistentemente a ClinicalBERT.")

EFECTO AISLADO DE LA VENTANA DE CONTEXTO
(misma arquitectura, mismos datos, solo cambia cuanto texto ve)

  exactitud    completo 0.715 -> truncado 0.575 (-0.140, -19.6%)
  f1_macro     completo 0.459 -> truncado 0.330 (-0.129, -28.1%)
  kappa        completo 0.544 -> truncado 0.315 (-0.229, -42.2%)

Conclusion: parte sustancial de la desventaja del transformer
no viene de la arquitectura sino de la ventana de 512 tokens.
Coincide con Li et al. (2022): Clinical-Longformer, que extiende
la ventana a 4,096, supera consistentemente a ClinicalBERT.


## 8. Validación con evaluador independiente

Las etiquetas de MIMIC derivan de codificación administrativa: son un
**estándar de plata**. Para medir el criterio contra juicio humano se realizó
una revisión ciega con dos anotadores independientes.

Controles aplicados:

- La interfaz oculta el veredicto del sistema, el código diagnóstico de origen
  y el estrato de muestreo.
- Orden de presentación aleatorio.
- **Registro automático del tiempo dedicado a cada caso** (auditable).
- Manual de anotación con regla de ancla: todo veredicto positivo exige cita
  literal del fragmento que lo sustenta.

In [12]:
if kap and kap.get("n_comunes"):
    p, b = kap["principal_3clases"], kap["binario"]
    print(f"Casos doble-anotados     : {kap['n_comunes']}")
    print(f"Acuerdo observado (Po)   : {p['po']:.3f}")
    print(f"Kappa de Cohen           : {p['puntual']:.3f}  "
          f"IC95 [{p['ic_bajo']:.3f}, {p['ic_alto']:.3f}]")
    print(f"PABAK                    : {p['pabak']:.3f}")
    print(f"Indice de prevalencia    : {b['indice_prevalencia']:.3f}")
    print(f"Indice de sesgo          : {b['indice_sesgo']:.3f}")
    print(f"McNemar (sesgo entre anotadores): p = {b['mcnemar_p']:.3f}")
    print()
    print("Interpretacion (Landis y Koch):", p["interpretacion"])
else:
    print("concordancia aun no disponible")

Casos doble-anotados     : 78
Acuerdo observado (Po)   : 0.872
Kappa de Cohen           : 0.644  IC95 [0.413, 0.817]
PABAK                    : 0.744
Indice de prevalencia    : 0.538
Indice de sesgo          : 0.051
McNemar (sesgo entre anotadores): p = 0.344

Interpretacion (Landis y Koch): sustancial


### Por qué se reporta el PABAK junto al kappa

Con prevalencias muy desiguales el kappa se deprime aunque el acuerdo observado
sea alto: es la **paradoja de Feinstein–Cicchetti (1990)**. Byrt et al. (1993)
propusieron el PABAK, que corrige el efecto de prevalencia y sesgo.

Reportar solo el kappa subestimaría la concordancia; reportar solo el PABAK la
sobreestimaría. La práctica correcta es presentar ambos junto al acuerdo
observado.

## 9. Transferencia al español con etiqueta de oro

Segundo corpus, en español y sobre la taxonomía peruana: ocurrencias
notificadas al sistema institucional de reporte, codificadas **una a una por
profesionales expertos** contra los Anexos 02 y 03 (tipo de evento, naturaleza
y severidad). Es un **estándar de oro**: juicio humano especializado.

Tres tareas, dos de ellas imposibles sobre MIMIC:

- **T1 evento adverso vs. incidente** — la distinción central de la norma (¿hubo
  daño?). No es medible en MIMIC porque una epicrisis no documenta cuasi-fallas.
- **T2 naturaleza** — homóloga a la Etapa 2, pero con etiqueta humana.
- **T3 severidad** — alimenta la matriz de priorización institucional.

In [13]:
if oe5:
    li = oe5["limpieza"]
    print("PREPROCESAMIENTO")
    print(f"  registros originales        : {li['filas_originales']:,}")
    print(f"  identificadores anonimizados: {li['dni_anonimizados']}")
    print(f"  textos demasiado breves     : {li['textos_cortos']}")
    print(f"  tras deduplicar             : {li['filas_unicas']:,}")
    print(f"  (se eliminaron {li['filas_originales']-li['filas_unicas']:,} "
          f"duplicados: sin este paso el mismo texto caia en train y test)")
    print()
    filas = []
    for nom, t in oe5["tareas"].items():
        mej = max((k for k in t["resultados"] if k != "por_clase"),
                  key=lambda k: t["resultados"][k]["f1_macro"])
        r = t["resultados"][mej]
        filas.append({"Tarea": nom.split("_",1)[1].replace("_"," ").capitalize(),
                      "n": t["n"], "Clases": t["clases"], "Modelo": mej,
                      "F1-macro": r["f1_macro"], "F1-micro": r["f1_micro"]})
    print(pd.DataFrame(filas).to_string(index=False))

PREPROCESAMIENTO
  registros originales        : 8,799
  identificadores anonimizados: 124
  textos demasiado breves     : 640
  tras deduplicar             : 6,336
  (se eliminaron 2,463 duplicados: sin este paso el mismo texto caia en train y test)

     Tarea    n  Clases    Modelo  F1-macro  F1-micro
   Binaria 6334       2    LogReg    0.8597    0.8642
Naturaleza 6276       9 LinearSVC    0.7675    0.8447
 Severidad 6279       4    LogReg    0.6063    0.7253


In [14]:
if oe5:
    pc = oe5["tareas"]["T2_naturaleza"]["resultados"]["por_clase"]
    d = (pd.DataFrame([{"Naturaleza": c.title(), "n": v["n"], "F1": v["f1"]}
                       for c, v in pc.items()])
           .sort_values("F1", ascending=False))
    print("F1 POR NATURALEZA (Anexo 02)")
    print(d.to_string(index=False))
    print()
    print("Gestion de la organizacion es el caso relevante: en MIMIC era")
    print("inviable (n=24, F1=0.000) y aqui alcanza F1 alto con ~1,431 ejemplos.")
    print("El corpus en espanol rescata clases que el corpus en ingles no cubre.")

F1 POR NATURALEZA (Anexo 02)
                        Naturaleza   n     F1
              Cuidado Del Paciente 454 0.9231
  Infección Asociada A La Atención  58 0.8793
                        Medicación 147 0.8754
                  Historia Clínica  42 0.8537
        Gestión De La Organización 286 0.8532
Dispositivo Médico / Equipo / Bien 134 0.7068
                           Insumos  56 0.6789
                     Procedimiento  44 0.5946
                    Comportamiento  35 0.5424

Gestion de la organizacion es el caso relevante: en MIMIC era
inviable (n=24, F1=0.000) y aqui alcanza F1 alto con ~1,431 ejemplos.
El corpus en espanol rescata clases que el corpus en ingles no cubre.


> **Advertencia de comparabilidad.** Estas cifras **no** son comparables con las
> de la sección 6. El texto del ERSP son descripciones de ~120 caracteres
> escritas por quien **ya identificó** el evento; la epicrisis son ~17,500
> caracteres donde el evento hay que **encontrarlo**. La tarea es más fácil por
> construcción, y su mejor rendimiento no indica un modelo superior sino un
> problema distinto.

## 10. Conclusiones

1. **Un modelo léxico bien construido superó al transformer clínico** en esta
   tarea. La hipótesis inicial se refutó y el resultado negativo se reporta
   como tal.

2. **Buena parte de esa diferencia se explica por la ventana de contexto**, no
   por la arquitectura. Truncar el texto a lo que ve BERT degrada al modelo
   léxico de forma sustancial con los mismos datos y el mismo algoritmo. La vía
   de mejora es una arquitectura de secuencia larga (Clinical-Longformer), no un
   preentrenamiento clínico más específico.

3. **El aporte metodológico principal es la auditoría de validez**: siete modos
   de fallo del etiquetado por reglas, con el confusor de época como caso
   ejemplar de aprendizaje por atajo. Un AUC de 0.973 puede sostenerse en el
   reconocimiento de una plantilla documental.

4. **La evaluación en cascada** muestra que reportar la clasificación aislada
   sobreestima el rendimiento operativo en torno a un tercio.

5. **La concordancia con un evaluador independiente** se sitúa en la banda
   sustancial, con la reserva del tamaño muestral.

6. **El corpus en español con etiqueta de oro** abre la transferencia a la
   taxonomía nacional, que es el destino aplicado del trabajo.

### Limitaciones declaradas

- **Estándar de plata:** las métricas sobre MIMIC miden acuerdo con códigos CIE,
  no con juicio clínico.
- **Reutilización del conjunto de prueba** a lo largo de las iteraciones del
  pipeline (sesgo optimista no cuantificado); el umbral no se ajustó.
- **Alcance de la validación experta:** la muestra está restringida a eventos de
  infección.
- **Cobertura de datos estructurados:** las tablas de UCI cubren el 19.7 % de las
  epicrisis.
- **Independencia del anotador:** el autor es a la vez desarrollador y anotador;
  se mitiga con interfaz ciega, orden aleatorio, cronometraje auditable y la
  medición del acuerdo contra un evaluador independiente.

---

*Repositorio: https://github.com/carlosperez100/PLN_SP ·
Reporte en línea: https://carlosperez100.github.io/PLN_SP/*